In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Entrées et sorties Estimator

<Accordion>
<AccordionItem title="Versions des packages">

Le code sur cette page a été développé en utilisant les exigences suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Cette page donne un aperçu des entrées et des sorties de la primitive Qiskit Runtime Estimator, qui exécute des charges de travail sur les ressources de calcul IBM Quantum&reg;. Estimator te permet de définir efficacement des charges de travail vectorisées en utilisant une structure de données appelée un [**Bloc Unifié de Primitive (PUB)**](/guides/primitive-input-output#pubs). Ils sont utilisés comme entrées pour la méthode [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) pour la primitive Estimator, qui exécute la charge de travail définie comme un job. Ensuite, une fois le job terminé, les résultats sont renvoyés dans un format qui dépend à la fois des PUBs utilisés et des options d'exécution spécifiées par la primitive.
## Entrées
Chaque PUB est dans ce format :

(`<circuit unique>`, `<un ou plusieurs observables>`, `<valeurs de paramètres optionnelles>`, `<précision optionnelle>`),

Les `valeurs de paramètres` optionnelles peuvent être une liste ou un seul paramètre. Les éléments des observables et des valeurs de paramètres sont combinés en suivant les règles de diffusion NumPy telles que décrites dans le sujet [Entrées et sorties des primitives](primitive-input-output#broadcasting-rules), et une estimation de la valeur d'espérance est renvoyée pour chaque élément de la forme diffusée.

> **Note:** Si l'entrée contient des mesures, elles sont ignorées.

Pour la primitive Estimator, un PUB peut contenir au maximum quatre valeurs :
- Un seul `QuantumCircuit`, qui peut contenir un ou plusieurs objets [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
- Une liste d'un ou plusieurs observables, qui spécifient les valeurs d'espérance à estimer, arrangées dans un tableau (par exemple, un seul observable représenté sous forme de tableau 0-d, une liste d'observables sous forme de tableau 1-d, etc.). Les données peuvent être dans l'un des formats `ObservablesArrayLike` comme `Pauli`, `SparsePauliOp`, `PauliList` ou `str`.
   > **Note:** - Les observables commutatifs **dans le même PUB** sont regroupés ensemble en utilisant [cette méthode](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting).
>    - Les observables commutatifs dans différents PUBs, même s'ils ont le même circuit, ne sont pas estimés en utilisant la même mesure. Chaque PUB représente une base différente pour la mesure, et par conséquent, des mesures séparées sont requises pour chaque PUB.
>    - Pour s'assurer que les observables commutatifs sont estimés en utilisant la même mesure, regroupe-les dans le même PUB.
- Une collection de valeurs de paramètres à lier au circuit. Cela peut être spécifié comme un seul objet de type tableau où le dernier index est sur les objets `Parameter` du circuit ou omis (ou de manière équivalente, défini à `None`) si le circuit n'a pas d'objets `Parameter`.
- (Optionnellement) Une précision cible pour les valeurs d'espérance à estimer
---

Le code suivant démontre un exemple d'ensemble d'entrées vectorisées pour la primitive `Estimator` et les exécute sur un Backend IBM&reg; comme un seul objet `RuntimeJobV2`.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

## Sorties
Après qu'un ou plusieurs PUBs sont envoyés à une QPU pour exécution et qu'un job se termine avec succès, les données sont renvoyées sous forme d'objet conteneur [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) accessible en appelant la méthode `RuntimeJobV2.result()`.

Le `PrimitiveResult` contient une liste itérable d'objets [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) qui contiennent les résultats d'exécution pour chaque PUB.

Chaque élément de cette liste correspond à chaque PUB soumis à la méthode `run()` de la primitive (par exemple, un job soumis avec 20 PUBs renverra un objet `PrimitiveResult` qui contient une liste de 20 objets `PubResult`, un correspondant à chaque PUB).

Chaque `PubResult` pour la primitive Estimator contient au moins un tableau de valeurs d'espérance (`PubResult.data.evs`) et les écarts-types associés (soit `PubResult.data.stds` soit `PubResult.data.ensemble_standard_error` selon le `resilience_level` utilisé), mais peut contenir plus de données selon les options d'atténuation d'erreurs qui ont été spécifiées.

Chaque objet `PubResult` possède à la fois un attribut `data` et un attribut `metadata`.
    - L'attribut `data` est un [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) personnalisé qui contient les valeurs de mesure réelles, les écarts-types, etc.
    - Le `DataBin` a divers attributs selon la forme ou la structure du PUB associé ainsi que les options d'atténuation d'erreurs spécifiées par la primitive utilisée pour soumettre le job (par exemple, [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) ou [PEC](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)).
    - L'attribut `metadata` contient des informations sur les options d'exécution et d'atténuation d'erreurs utilisées (expliquées plus loin dans la section [Métadonnées des résultats](#result-metadata) de cette page).

Ce qui suit est un aperçu visuel de la structure de données `PrimitiveResult` pour la sortie Estimator :

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```

En termes simples, un seul job renvoie un objet `PrimitiveResult` et contient une liste d'un ou plusieurs objets `PubResult`. Ces objets `PubResult` stockent ensuite les données de mesure pour chaque PUB qui a été soumis au job.

L'extrait de code ci-dessous décrit le format `PrimitiveResult` (et le `PubResult` associé) pour le job créé ci-dessus.

In [2]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
with np.printoptions(threshold=200):
    print(
        f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
    )

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

#### How the Estimator primitive calculates error

In addition to the estimate of the mean of the observables passed in the input PUBs (the `evs` field of the `DataBin`), Estimator also attempts to deliver an estimate of the error associated with those expectation values. All Estimator queries will populate the `stds` field with a quantity like the standard error of the mean for each expectation value, but some error mitigation options produce additional information, such as `ensemble_standard_error`.

Consider a single observable $\mathcal{O}$. In the absence of [ZNE](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), you can think of each shot of the Estimator execution as providing a point estimate of the expectation value $\langle \mathcal{O} \rangle$. If the pointwise estimates are in a vector `Os`, then the value returned in `ensemble_standard_error` is equivalent to the following (in which $\sigma_{\mathcal{O}}$ is the [standard deviation of the expectation value](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) estimate and $N_{shots}$ is the number of shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

which treats all shots as part of a single ensemble. If you requested gate [twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), you can sort the pointwise estimates of $\langle \mathcal{O} \rangle$ into sets that share a common twirl. Call these sets of estimates `O_twirls`, and there are `num_randomizations` (number of twirls) of them. Then `stds` is the standard error of the mean of `O_twirls`, as in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

where $\sigma_{\mathcal{O}}$ is the standard deviation of `O_twirls` and $N_{twirls}$ is the number of twirls. When you do not enable twirling, `stds` and `ensemble_standard_error` are equal.

If you enable ZNE, then the `stds` described above become weights in a non-linear regression to an extrapolator model. What finally gets returned in the `stds` in this case is the uncertainty of the fit model evaluated at a noise factor of zero. When there is a poor fit, or large uncertainty in the fit, the reported `stds` can become very large. When ZNE is enabled, `pub_result.data.evs_noise_factors` and `pub_result.data.stds_noise_factors` are also populated, so that you can do your own extrapolation.

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [3]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'dynamical_decoupling' : {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'},
'twirling' : {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'},
'resilience' : {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False},
'version' : 2,

The metadata of the PubResult result is:
'shots' : 4096,
'target_precision' : 0.015625,
'circuit_metadata' : {},
'resilience' : {},
'num_randomizations' : 32,
